# MeterIQ — Smart Meter Analytics: EDA & Preprocessing
### Mathura and Bareilly Electricity Consumption Data
---

This is the first analysis notebook for the MeterIQ college project. It studies and preprocesses the supplied smart-meter data before any feature engineering, anomaly detection, theft-risk scoring, or forecasting.

**Important:** findings in this notebook are data-quality observations only. No result is presented as proof of electricity theft.

## Table of Contents

1. Environment setup and configuration  
2. Dataset inventory and schema validation  
3. Data-quality report  
4. Consumption and meter-level exploration  
5. Reproducible preprocessing pipeline  
6. Post-cleaning validation and output files  
7. Findings and next steps

## 1. Environment Setup & Configuration

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import re

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda number: f'{number:,.4f}')

ZIP_PATH = Path('/Users/yashitarora/Downloads/archive (7).zip')
OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Archive not found: {ZIP_PATH}. Update ZIP_PATH before continuing.')

## 2. Dataset Inventory & Schema Validation

In [ ]:
with ZipFile(ZIP_PATH) as archive:
    inventory = pd.DataFrame([
        {'file': item.filename, 'size_mb': round(item.file_size / 1_048_576, 2)}
        for item in archive.infolist() if item.filename.lower().endswith('.csv')
    ])

display(inventory)
print('Observed raw-file fields: x_Timestamp, t_kWh, z_Avg Voltage (Volt), z_Avg Current (Amp), y_Freq (Hz), meter')
print('Observed aggregated-file fields: meter, Date, t_kWh')

In [ ]:
# Select one source file. Start with an Aggregated file; run again for each dataset.
DATASET_NAME = 'SM Cleaned Data BR Aggregated.csv'
ROWS_TO_READ = None  # Set e.g. 100_000 only for a dry run on a large raw file.

if DATASET_NAME not in set(inventory['file']):
    raise ValueError('Choose a dataset name exactly as displayed in the inventory.')

with ZipFile(ZIP_PATH) as archive:
    with archive.open(DATASET_NAME) as source:
        raw = pd.read_csv(source, nrows=ROWS_TO_READ, low_memory=False)

TIME_SOURCE = 'Date' if 'Date' in raw.columns else 'x_Timestamp'
REQUIRED = {'meter', TIME_SOURCE, 't_kWh'}
if missing := REQUIRED - set(raw.columns):
    raise ValueError(f'Unsupported or changed schema; required fields missing: {sorted(missing)}')

print(f'Loaded rows: {len(raw):,}')
print('Validated meter field: meter')
print(f'Validated timestamp/date field: {TIME_SOURCE}')
print('Validated consumption field: t_kWh')
raw.head()

## 3. Data Quality Report

In [ ]:
def make_quality_report(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'data_type': frame.dtypes.astype(str),
        'missing_count': frame.isna().sum(),
        'missing_percent': (frame.isna().mean() * 100).round(3),
        'unique_values': frame.nunique(dropna=True),
    })

display(make_quality_report(raw))
print(f'Exact duplicate rows: {raw.duplicated().sum():,}')
print(f'Unique meters: {raw["meter"].nunique(dropna=True):,}')

In [ ]:
# Inspect parsed timestamps and consumption before applying any cleaning rule.
inspection = raw[['meter', TIME_SOURCE, 't_kWh']].copy()
inspection['parsed_time'] = pd.to_datetime(inspection[TIME_SOURCE], errors='coerce')
inspection['consumption_kwh'] = pd.to_numeric(inspection['t_kWh'], errors='coerce')

print('Unparseable timestamps:', inspection['parsed_time'].isna().sum())
print('Non-numeric consumption values:', inspection['consumption_kwh'].isna().sum())
print('Negative consumption values:', inspection['consumption_kwh'].lt(0).sum())
print('Zero consumption values:', inspection['consumption_kwh'].eq(0).sum())
print('Date range:', inspection['parsed_time'].min(), 'to', inspection['parsed_time'].max())
inspection['consumption_kwh'].describe()

## 4. Consumption & Meter-Level Exploration

In [ ]:
# Readings are ordered separately per meter before sampling frequency is calculated.
ordered = inspection.dropna(subset=['meter', 'parsed_time']).sort_values(['meter', 'parsed_time'])
intervals = ordered.groupby('meter')['parsed_time'].diff().dropna()
print('Median observed interval:', intervals.median() if not intervals.empty else 'not enough timestamped records')
display(intervals.value_counts().head(10).rename('record_count').to_frame())

meter_summary = (inspection.groupby('meter')['consumption_kwh']
                 .agg(records='size', total_kwh='sum', mean_kwh='mean', min_kwh='min', max_kwh='max')
                 .sort_values('total_kwh', ascending=False))
display(meter_summary.head(15))

In [ ]:
# Visual inspection: top meter's consumption history and distribution.
SELECTED_METER = meter_summary.index[0]
selected = inspection.loc[inspection['meter'].eq(SELECTED_METER)].sort_values('parsed_time')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].plot(selected['parsed_time'], selected['consumption_kwh'], linewidth=0.8)
axes[0].set(title=f'Consumption history — {SELECTED_METER}', xlabel='Time', ylabel='kWh')
axes[1].hist(inspection['consumption_kwh'].dropna(), bins=50, color='#2878B5')
axes[1].set(title='Consumption distribution', xlabel='kWh per reading', ylabel='Records')
plt.tight_layout()
plt.show()

## 5. Reproducible Preprocessing Pipeline

**Policy:** preserve the archive unchanged; standardize names only in a working copy; remove only exact duplicates from the accepted output; route missing meter/time/consumption and negative consumption to a separate audit output. The dataset documentation should be consulted before deciding whether negative readings have a valid domain meaning.

In [ ]:
def preprocess_smart_meter_data(frame: pd.DataFrame, time_source: str):
    working = frame.copy()
    working['meter_id'] = working['meter'].astype('string').str.strip()
    working['timestamp'] = pd.to_datetime(working[time_source], errors='coerce')
    working['consumption_kwh'] = pd.to_numeric(working['t_kWh'], errors='coerce')

    sensor_map = {
        'z_Avg Voltage (Volt)': 'avg_voltage_volt',
        'z_Avg Current (Amp)': 'avg_current_amp',
        'y_Freq (Hz)': 'frequency_hz',
    }
    for source, destination in sensor_map.items():
        if source in working.columns:
            working[destination] = pd.to_numeric(working[source], errors='coerce')

    flags = pd.DataFrame(index=working.index)
    flags['exact_duplicate'] = working.duplicated(keep='first')
    flags['missing_meter_id'] = working['meter_id'].isna() | working['meter_id'].eq('')
    flags['invalid_timestamp'] = working['timestamp'].isna()
    flags['invalid_consumption'] = working['consumption_kwh'].isna()
    flags['negative_consumption'] = working['consumption_kwh'].lt(0).fillna(False)

    reject_mask = flags.any(axis=1)
    rejected = working.loc[reject_mask].copy()
    rejected['rejection_reasons'] = flags.loc[reject_mask].apply(
        lambda row: '; '.join(row.index[row].tolist()), axis=1
    )
    clean = (working.loc[~reject_mask]
             .sort_values(['meter_id', 'timestamp'])
             .reset_index(drop=True))
    return clean, rejected, flags

clean, rejected, flags = preprocess_smart_meter_data(raw, TIME_SOURCE)

preprocessing_audit = flags.sum().rename('records').to_frame()
preprocessing_audit['percent_of_loaded_rows'] = (preprocessing_audit['records'] / len(raw) * 100).round(3)
display(preprocessing_audit)
print(f'Accepted records: {len(clean):,}')
print(f'Rejected records: {len(rejected):,}')

## 6. Post-Cleaning Validation & Output Files

In [ ]:
same_meter_timestamp = clean.duplicated(['meter_id', 'timestamp'], keep=False).sum()
print('Duplicate meter/timestamp records retained for review:', f'{same_meter_timestamp:,}')
print('Clean dataset date range:', clean['timestamp'].min(), 'to', clean['timestamp'].max())
display(clean[['meter_id', 'timestamp', 'consumption_kwh']].head())

slug = re.sub(r'[^a-z0-9]+', '_', Path(DATASET_NAME).stem.lower()).strip('_')
clean_path = OUTPUT_DIR / f'{slug}_processed.csv'
rejected_path = OUTPUT_DIR / f'{slug}_rejected.csv'
report_path = OUTPUT_DIR / f'{slug}_preprocessing_report.json'

clean.to_csv(clean_path, index=False)
rejected.to_csv(rejected_path, index=False)
report_path.write_text(json.dumps({
    'source_archive': str(ZIP_PATH), 'source_file': DATASET_NAME,
    'loaded_rows': int(len(raw)), 'accepted_rows': int(len(clean)), 'rejected_rows': int(len(rejected)),
    'cleaning_counts': {name: int(value) for name, value in flags.sum().items()},
    'same_meter_timestamp_records': int(same_meter_timestamp),
    'note': 'Raw data was not altered. Rejected rows are retained separately for audit.'
}, indent=2), encoding='utf-8')

print('Processed data:', clean_path)
print('Rejected-record audit:', rejected_path)
print('Preprocessing report:', report_path)

## 7. Findings & Next Steps

- The archive provides both raw interval readings and daily aggregated datasets for Mathura and Bareilly.
- The observed meter identifier is `meter`; consumption is `t_kWh`; time is either `x_Timestamp` or `Date`.
- This notebook must be run once per selected source file to produce the real quality counts and cleaned output.
- Review the audit files before modeling. Repeated meter/timestamp records are reported, not automatically deleted.
- The next notebook should use the cleaned output to create time-aware features (calendar features, lags, and rolling statistics) without data leakage.